## Read raw data from volume "source_systems" and write into bronze as delta tables

### 00. Import required library

In [0]:
from pyspark.sql.functions import *
from datetime import datetime

### 01. Generate date-specific schema name

In [0]:
ingest_date = datetime.today().strftime("%Y-%m-%d")
schema_date = datetime.today().strftime("bronze_%Y_%m_%d")

catalog = "`abc_business-core`"   # FIXED: back-ticked
full_schema = f"{catalog}.{schema_date}"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {full_schema}")
print(f"Using schema: {full_schema}")

### 02. Define volume paths

In [0]:
crm_path = "/Volumes/abc_business-core/bronze/source_systems/source_crm"
erp_path = "/Volumes/abc_business-core/bronze/source_systems/source_erp"

### 03. Helper function to load CSVs

In [0]:
def load_csv(path):
    return (
        spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv(path)
            .withColumn("ingest_date", lit(ingest_date))
    )

### 04. Load CRM datasets

In [0]:
df_customers = load_csv(f"{crm_path}/cust_info.csv")
df_products  = load_csv(f"{crm_path}/prd_info.csv")
df_sales     = load_csv(f"{crm_path}/sales_details.csv")

### 05. Load ERP datasets

In [0]:
df_erp_customers = load_csv(f"{erp_path}/CUST_AZ12.csv")
df_erp_locations = load_csv(f"{erp_path}/LOC_A101.csv")
df_erp_pricing   = load_csv(f"{erp_path}/PX_CAT_G1V2.csv")

### 06. Write to date-specific Bronze schema

In [0]:
df_customers.write.format("delta").mode("overwrite").saveAsTable(f"{full_schema}.customers_info")
df_products.write.format("delta").mode("overwrite").saveAsTable(f"{full_schema}.product_info")
df_sales.write.format("delta").mode("overwrite").saveAsTable(f"{full_schema}.sales_details")

df_erp_customers.write.format("delta").mode("overwrite").saveAsTable(f"{full_schema}.erp_customers")
df_erp_locations.write.format("delta").mode("overwrite").saveAsTable(f"{full_schema}.erp_locations")
df_erp_pricing.write.format("delta").mode("overwrite").saveAsTable(f"{full_schema}.erp_pricing")

print(f"Bronze ingestion completed into schema: {full_schema}")